# CardioIA — Fase 6 · Treinamento do modelo de `pico_risco`

## 1. Objetivo do experimento

**Objetivo.** Treinar, avaliar e salvar um classificador supervisionado capaz de prever a
variável `pico_risco` a partir da base **sintética** do material oficial da aula
(`data/base_dados.py`).

A partir da Etapa 2, **a lógica de Machine Learning vive em módulos Python reutilizáveis**
(`src/cardioia/ml/`) e este notebook os consome. Assim o mesmo código é usado pelo notebook,
pela suíte de testes (`pytest`) e, na próxima etapa, pelos agentes do OpenAI Agents SDK — sem
duas implementações da mesma regra.

| Módulo | Papel |
|---|---|
| `cardioia.ml.dataset` | carrega e valida a base oficial; separa `X`/`y`; regra determinística do rótulo |
| `cardioia.ml.evaluate` | acurácia, matriz de confusão, relatório e métricas serializáveis |
| `cardioia.ml.train` | treina, compara pelo critério pré-declarado e salva os artefatos (também via CLI) |
| `cardioia.ml.predict` | carrega o modelo salvo e prevê `pico_risco` (contrato validado) |

**O que este notebook faz, na ordem:**

| # | Seção |
|---|---|
| 1 | Objetivo do experimento |
| 2 | Verificação das dependências |
| 3 | Bootstrap do pacote, imports e semente aleatória |
| 4 | Carregamento/geração dos dados conforme o material oficial |
| 5 | Inspeção da base (dimensões, tipos, nulos, distribuição do alvo) |
| 6 | Variáveis preditoras e variável-alvo (`pico_risco`) |
| 7 | Separação treino/teste com estratificação |
| 8 | Treinamento de classificadores supervisionados |
| 9 | Avaliação: acurácia, matriz de confusão e relatório de classificação |
| 10 | Análise dos resultados, divergências e limitações |
| 11 | Salvamento do modelo treinado |
| 12 | Carregamento do modelo salvo e previsão de um novo caso |
| 13 | Simulação com os dados do paciente fornecidos pela aula |
| 14 | Persistência dos artefatos no Google Colab |

**Como executar no Google Colab**

1. Em **Google Drive**, compacte a pasta do projeto (`cardioia_Fase6/`) em `.zip`.
2. Abra <https://colab.research.google.com/> → `Arquivo ▸ Fazer upload de notebook` → este arquivo.
3. Rode a célula abaixo para enviar e descompactar o projeto (ela **precisa** de `data/` e `src/`):

   ```python
   from google.colab import files
   import zipfile

   enviado = files.upload()                     # selecione cardioia_Fase6.zip
   with zipfile.ZipFile(next(iter(enviado))) as pacote:
       pacote.extractall("/content")            # cria /content/cardioia_Fase6/
   ```

4. `Ambiente de execução ▸ Executar tudo`. A célula 3.1 localiza a raiz do projeto e insere
   `src/` no `sys.path` automaticamente.
5. A seção 14 empacota `models/` em `.zip` e baixa os artefatos.

> Alternativa: `!git clone https://github.com/ReMendess/cardioia_Fase6.git` — só funciona depois que
> `data/` e `src/` estiverem commitados/pushados no repositório. Hoje o clone **não** traz a pasta
> `data/` (ela ainda está fora do controle de versão), por isso o caminho recomendado é o `.zip`.

**Execução local** (a partir da raiz do repositório):

```powershell
python -m pip install -r requirements.txt
python -m nbconvert --to notebook --execute notebooks/treinamento_modelo.ipynb
python -m pytest tests -q                 # 73 testes da camada de ML
python -m src.cardioia.ml.train            # treina e grava models/
```

> **Aviso de responsabilidade.** Os dados são **sintéticos**, gerados por regras, e o rótulo
> `pico_risco` é uma função determinística de 6 variáveis. Métricas altas neste notebook
> significam que o modelo reconstruiu essa regra — **não** constituem validação clínica, não
> usam dados de pacientes reais e não devem embasar decisões médicas.

## 2. Dependências

O notebook precisa apenas de `numpy`, `pandas`, `scikit-learn` e `joblib`. O Google Colab já traz
todas instaladas; a célula abaixo **verifica** e só instala o que faltar (nada é baixado se o
ambiente já estiver completo). Nenhuma dependência nova foi adicionada na Etapa 2.

In [ ]:
# 2.1 - Verificacao das dependencias (sem instalacoes superfluas)
import importlib
import subprocess
import sys

REQUISITOS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

faltando = []
for modulo in REQUISITOS:
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltando.append(modulo)

if faltando:
    pacotes = [REQUISITOS[modulo] for modulo in faltando]
    print("Instalando dependencias ausentes:", pacotes)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pacotes], check=False)
else:
    print("Todas as dependencias ja estao disponiveis - nenhuma instalacao necessaria.")

import joblib
import numpy
import pandas
import sklearn

EM_COLAB = "google.colab" in sys.modules
print("python  :", sys.version.split()[0])
print("numpy   :", numpy.__version__)
print("pandas  :", pandas.__version__)
print("sklearn :", sklearn.__version__)
print("joblib  :", joblib.__version__)
print("ambiente:", "Google Colab" if EM_COLAB else "local")

## 3. Bootstrap do pacote, imports e semente aleatória

A célula 3.1 localiza a **raiz do projeto** (a pasta que contém `data/base_dados.py` e
`src/cardioia/ml/dataset.py`), insere `src/` no `sys.path` e importa os módulos. Nenhum caminho
absoluto da máquina do autor: o notebook funciona localmente e no Colab.

Todos os parâmetros do experimento vêm dos módulos (fonte única de verdade):

- `SEMENTE = 42` — igual à semente do gerador oficial;
- `TAMANHO_TESTE = 0.20` — divisão estratificada 80/20;
- `COLUNAS_PREDITORAS` — as 10 features, na ordem do arquivo oficial;
- `LIMIARES_DO_ROTULO` e `CORTE_RISK_SCORE` — usados apenas para checagem e para o oráculo
  determinístico. **Nada disso entra em `X`** (isso seria vazamento do alvo).

In [ ]:
# 3.1 - Bootstrap do pacote cardioia + parametros do experimento
from pathlib import Path

NOME_DO_REPOSITORIO = "cardioia_Fase6"
URL_DO_REPOSITORIO = "https://github.com/ReMendess/cardioia_Fase6.git"


def parece_raiz_do_projeto(caminho: Path) -> bool:
    # A raiz precisa ter o material oficial E a camada modularizada.
    return (caminho / "data" / "base_dados.py").is_file() and (
        caminho / "src" / "cardioia" / "ml" / "dataset.py"
    ).is_file()


def localizar_raiz(inicio=None):
    base = Path(inicio or Path.cwd()).resolve()
    for candidato in (base, *base.parents):
        if parece_raiz_do_projeto(candidato):
            return candidato
    return None


RAIZ = localizar_raiz()

if RAIZ is None:
    # Plano B: clonar o repositorio (funciona apenas se data/ e src/ estiverem versionados).
    destino = Path("/content") if Path("/content").is_dir() else Path.cwd()
    print("[bootstrap] projeto nao encontrado no ambiente; tentando clonar o repositorio...")
    subprocess.run(
        ["git", "clone", "--depth", "1", URL_DO_REPOSITORIO, str(destino / NOME_DO_REPOSITORIO)],
        check=False,
    )
    RAIZ = localizar_raiz(destino / NOME_DO_REPOSITORIO)

if RAIZ is None:
    raise RuntimeError(
        "Projeto CardioIA nao encontrado. Envie a pasta do projeto antes de continuar:\n"
        "  1) compacte 'cardioia_Fase6/' em .zip;\n"
        "  2) no Colab, rode:  from google.colab import files; files.upload()\n"
        "     e descompacte com zipfile.ZipFile('<arquivo>.zip').extractall('/content')\n"
        "Localmente, execute o notebook a partir da raiz do repositorio."
    )

PASTA_SRC = RAIZ / "src"
if str(PASTA_SRC) not in sys.path:
    sys.path.insert(0, str(PASTA_SRC))

from cardioia.ml import dataset, evaluate, predict, train  # noqa: E402 - depende do sys.path

SEMENTE = dataset.SEMENTE
TAMANHO_TESTE = dataset.TAMANHO_TESTE
COLUNAS_PREDITORAS = list(dataset.COLUNAS_PREDITORAS)
LIMIARES_DO_ROTULO = dict(dataset.LIMIARES_DO_ROTULO)
CORTE_RISK_SCORE = dataset.CORTE_RISK_SCORE
PASTA_MODELOS = train.resolver_pasta_modelos(raiz=RAIZ)

print("Diretorio atual     :", Path.cwd())
print("Raiz do projeto     :", RAIZ)
print("Pasta de modelos    :", PASTA_MODELOS)
print("Semente             :", SEMENTE, "| tamanho do teste:", TAMANHO_TESTE)
print("Features            :", len(COLUNAS_PREDITORAS))
print("Modulos importados  :", ", ".join(["dataset", "evaluate", "predict", "train"]))

## 4. Carregamento ou geração dos dados conforme o material oficial

O material da aula fornece um **gerador** (`data/base_dados.py`), não uma base tabular pronta.
O módulo `cardioia.ml.dataset` é a autoridade sobre esse arquivo: ele o executa em um namespace
isolado, **sem editá-lo**, e valida o resultado.

Ordem de resolução (failover), sempre registrada no `metadata` do modelo:

1. `data/base_dados.py` presente no ambiente (caminho normal, usado quando o zip/clone está lá);
2. **réplica inline idêntica ao oficial**, passada explicitamente como `codigo_fonte=` — só é
   acionada aqui, no notebook, e imprime aviso de que houve failover.

O módulo **não** usa réplica por conta própria: se o arquivo oficial não existir, ele levanta um
erro claro (`ArquivoOficialNaoEncontradoError`) pedindo a pasta do projeto. Assim nenhum resultado
apresentado como "do material oficial" é, na verdade, de uma cópia paralela.

Além da base, lemos também `data/base_conhecimento.py` (protocolos) e `data/novo_paciente.py`
(caso de simulação), preservados como estão.

In [ ]:
# 4.1 - Leitura dos tres arquivos oficiais de data/ (preservados, sem edicao)
ARQUIVOS_OFICIAIS = (
    dataset.NOME_ARQUIVO_BASE,
    dataset.NOME_ARQUIVO_CONHECIMENTO,
    dataset.NOME_ARQUIVO_NOVO_PACIENTE,
)

fontes = {}
for nome in ARQUIVOS_OFICIAIS:
    try:
        fontes[nome] = dataset.ler_arquivo_oficial(nome, raiz=RAIZ)
        print(f"[OK]      data/{nome} | {len(fontes[nome])} caracteres")
    except dataset.ArquivoOficialNaoEncontradoError as erro:
        print(f"[AUSENTE] {erro}")

print()
print("Base oficial em:", dataset.caminho_da_base_oficial(raiz=RAIZ))

In [ ]:
# 4.2 - Geracao da base pelo modulo (com failover explicito e documentado)
# Replica inline do arquivo data/base_dados.py, usada SOMENTE como failover deste notebook.
# O texto abaixo e identico ao material oficial da aula (mesma semente, mesmas distribuicoes,
# mesma regra de rotulo). O modulo nao usa replica por conta propria: ela e passada aqui.
REPLICA_BASE_DADOS = r'''
import numpy as np
import pandas as pd

np.random.seed(42)
n = 5000

df = pd.DataFrame({
    "idade_media": np.random.normal(55, 12, n).clip(18, 90),
    "carga_sistema": np.random.uniform(30, 100, n),
    "tempo_espera": np.random.normal(45, 20, n).clip(5, 180),
    "fc_media": np.random.normal(85, 15, n).clip(50, 150),
    "spo2_media": np.random.normal(95, 3, n).clip(70, 100),
    "pa_sistolica": np.random.normal(125, 20, n).clip(80, 200),
    "temperatura": np.random.normal(24, 5, n).clip(10, 40),
    "densidade_eventos": np.random.poisson(3, n),
    "eventos_24h": np.random.poisson(2, n),
    "disponibilidade_recursos": np.random.uniform(20, 100, n)
})

risk_score = (
    (df["carga_sistema"] > 80).astype(int) +
    (df["tempo_espera"] > 60).astype(int) +
    (df["spo2_media"] < 92).astype(int) +
    (df["fc_media"] > 110).astype(int) +
    (df["eventos_24h"] > 4).astype(int) +
    (df["disponibilidade_recursos"] < 40).astype(int)
)

df["pico_risco"] = (risk_score >= 2).astype(int)
'''

try:
    # Caminho normal: executa o arquivo oficial do material da aula.
    df, origem = dataset.carregar_base_oficial(raiz=RAIZ, retornar_origem=True)
except dataset.ArquivoOficialNaoEncontradoError as erro:
    print("[FAILOVER]", erro)
    print("[ATENCAO] A base NAO veio do arquivo oficial - registre este failover no relatorio.")
    df, origem = dataset.carregar_base_oficial(
        codigo_fonte=REPLICA_BASE_DADOS, retornar_origem=True
    )

origem_dados = origem["origem"]
print()
print("origem da base         :", origem_dados)
print("usado failover         :", origem["usado_failover"])
print("dimensoes do DataFrame :", df.shape)
print("sha256 da base         :", dataset.sha256_da_base(df))

In [ ]:
# 4.3 - Base de conhecimento (protocolos) e caso de simulacao do paciente
if dataset.NOME_ARQUIVO_CONHECIMENTO in fontes:
    espaco = {}
    exec(compile(fontes[dataset.NOME_ARQUIVO_CONHECIMENTO], "data/base_conhecimento.py", "exec"), espaco)
    PROTOCOLOS_AULA = list(espaco["protocolos"])
    print("Protocolos carregados de data/base_conhecimento.py:", len(PROTOCOLOS_AULA))
else:
    PROTOCOLOS_AULA = []
    print("[AVISO] data/base_conhecimento.py ausente - a secao 13 nao tera protocolos.")

for indice, protocolo in enumerate(PROTOCOLOS_AULA, start=1):
    print(f"{indice:2d}. {protocolo}")

print()
if dataset.NOME_ARQUIVO_NOVO_PACIENTE in fontes:
    fonte_paciente = fontes[dataset.NOME_ARQUIVO_NOVO_PACIENTE]
    print("Inspecao estatica de data/novo_paciente.py (arquivo preservado, nao editado):")
    print("  - contem 'import pandas':", "import pandas" in fonte_paciente)
    print("  - usa a coluna 'disponibilidade recursos' (com espaco):", "disponibilidade recursos" in fonte_paciente)
    print("  - usa a coluna 'disponibilidade_recursos' (com underscore):", "disponibilidade_recursos" in fonte_paciente)
    print("  => o fragmento da aula nao executa como esta (falta o import de pandas) e a")
    print("     ultima coluna esta escrita com espaco.")
else:
    print("[AVISO] data/novo_paciente.py ausente - a secao 13 usara o caso descrito no plano.")

# O modulo le o caso oficial com um 'pd' substituto e normaliza o nome da coluna.
caso_paciente = dataset.carregar_caso_do_paciente_da_aula(raiz=RAIZ)
print()
print("Caso do paciente normalizado (disponibilidade recursos -> disponibilidade_recursos):")
for nome, valor in caso_paciente.items():
    print(f"  {nome:28s}: {valor}")

## 5. Inspeção da base

Antes de modelar, verificamos o que a base realmente contém: dimensões, tipos, valores ausentes,
duplicatas, estatísticas descritivas, distribuição do alvo `pico_risco` e a distribuição do
`risk_score` que deu origem ao rótulo.

`dataset.validar_base` levanta erro (`BaseInvalidaError`) em base vazia, coluna ausente, tipo não
numérico, valor ausente ou alvo não binário — e reporta duplicatas/valores fora de faixa como
avisos. A reconstrução do rótulo (`risk_score >= 2`) serve como **checagem de integridade**: prova
que os limiares registrados correspondem ao arquivo oficial.

In [ ]:
# 5.1 - Validacao e inspecao da base (modulo dataset)
resumo = dataset.validar_base(df)
print("Dimensoes:", resumo["dimensoes"])
print()
print("Tipos, distintos e ausentes por coluna:")
for coluna in df.columns:
    ausentes = resumo["valores_ausentes_por_coluna"].get(coluna, 0)
    print(f"  {coluna:28s} {resumo['tipos'][coluna]:8s} distintos={df[coluna].nunique():5d} ausentes={ausentes}")
print()
print("Total de valores ausentes :", resumo["valores_ausentes_total"])
print("Linhas duplicadas         :", resumo["linhas_duplicadas"])
print("Avisos da validacao       :", resumo["avisos"] if resumo["avisos"] else "nenhum")
print()
print("Estatisticas descritivas:")
print(df.describe().round(3).to_string())

In [ ]:
# 5.2 - Distribuicao da variavel-alvo e reconstrucao deterministica do risk_score
contagens = df[dataset.VARIAVEL_ALVO].value_counts().sort_index()
proporcoes = df[dataset.VARIAVEL_ALVO].value_counts(normalize=True).sort_index()

distribuicao = pandas.DataFrame({"contagem": contagens, "proporcao": proporcoes.round(4)})
distribuicao.index.name = dataset.VARIAVEL_ALVO
print("Distribuicao de", dataset.VARIAVEL_ALVO + ":")
print(distribuicao.to_string())
print()
print("Classe minoritaria (pico_risco=1):", f"{proporcoes.get(1, 0):.2%}", "da base")
print("Leitura: ha desbalanceamento moderado. A acuracia isolada nao basta -")
print("por isso o recall e a precisao da classe 1 tambem serao reportados (secao 9).")
print()

# Regra oficial recalculada pelo modulo (mesma logica de data/base_dados.py).
risk_score_recalculado = dataset.calcular_risk_score(df)
pico_risco_recalculado = dataset.rotulo_pela_regra(df)

print("Distribuicao do risk_score reconstruido:")
print(risk_score_recalculado.value_counts().sort_index().to_string())
print()
concordancia = float((pico_risco_recalculado == df[dataset.VARIAVEL_ALVO]).mean())
print("Concordancia regra reconstruida x coluna pico_risco:", concordancia)
assert concordancia == 1.0, "A regra reconstruida NAO reproduz o rotulo oficial."
print("[OK] Os limiares do modulo correspondem exatamente ao material oficial.")
print()
print("Frequencia dos limiares TEXTUAIS da base de conhecimento da aula,")
print("que sao DIFERENTES dos limiares do rotulo:")
efeito_dos_protocolos = pandas.Series({
    "SPO2 < 88": int((df["spo2_media"] < 88).sum()),
    "carga do sistema > 85": int((df["carga_sistema"] > 85).sum()),
    "FC > 120": int((df["fc_media"] > 120).sum()),
    "eventos_24h > 5": int((df["eventos_24h"] > 5).sum()),
    "disponibilidade_recursos < 30": int((df["disponibilidade_recursos"] < 30).sum()),
    "tempo_espera > 90": int((df["tempo_espera"] > 90).sum()),
    "temperatura > 38": int((df["temperatura"] > 38).sum()),
    "densidade_eventos > 6": int((df["densidade_eventos"] > 6).sum()),
    "PA sistolica < 90": int((df["pa_sistolica"] < 90).sum()),
})
print(efeito_dos_protocolos.to_string())

## 6. Preparação das variáveis preditoras e da variável `pico_risco`

- **`X`**: as 10 colunas originais do gerador, na ordem em que aparecem no arquivo oficial.
- **`y`**: a coluna `pico_risco`.

**Por que não criar `risk_score` como feature?** Porque `pico_risco` é uma função determinística de
`risk_score` (`>= 2`): usá-lo (ou qualquer indicador derivado dele) como entrada seria vazamento do
alvo — o modelo acertaria por construção, sem aprender nada.

`dataset.separar_features_e_alvo` concentra essas garantias: confere o contrato de colunas, remove
qualquer coluna auxiliar derivada do alvo e recusa `X`/`y` com valores ausentes.

In [ ]:
# 6.1 - Separacao X/y pelo modulo + verificacoes anti-vazamento
X, y = dataset.separar_features_e_alvo(df)

assert set(COLUNAS_PREDITORAS).issubset(df.columns)
assert dataset.VARIAVEL_ALVO not in COLUNAS_PREDITORAS
assert "risk_score" not in X.columns, "Vazamento: coluna derivada do alvo entrou em X."
assert not [c for c in X.columns if "risk" in c]
assert X.isna().sum().sum() == 0 and y.isna().sum() == 0

print("X:", X.shape, "| y:", y.shape)
print("Features (ordem preservada do material oficial):")
for posicao, coluna in enumerate(X.columns, start=1):
    print(f"  {posicao:2d}. {coluna:28s} {str(X[coluna].dtype)}")
print()
print("Variavel-alvo:", dataset.VARIAVEL_ALVO, "| valores possiveis:", sorted(y.unique().tolist()))
print("Colunas derivadas do alvo excluidas:", list(dataset.COLUNAS_AUXILIARES_IGNORADAS))
print("Valores ausentes em X:", int(X.isna().sum().sum()), "| em y:", int(y.isna().sum()))

## 7. Separação entre treino e teste com estratificação

`dataset.divisao_treino_teste` é a **única** fonte da partição: 80% treino / 20% teste,
`random_state=42` e `stratify=y` (mantém a proporção de ~26,6% de positivos nos dois conjuntos).

Centralizar isso importa: se o notebook e os módulos usassem partições diferentes, qualquer
comparação numérica entre eles seria inválida. O teste é usado **apenas** para medir — nenhum
ajuste de modelo acontece depois de olhar o teste.

In [ ]:
# 7.1 - Divisao estratificada (modulo dataset)
X_treino, X_teste, y_treino, y_teste = dataset.divisao_treino_teste(X, y)

resumo_split = pandas.DataFrame({
    "amostras": [len(y_treino), len(y_teste)],
    "positivos": [int(y_treino.sum()), int(y_teste.sum())],
    "proporcao_positivos": [float(y_treino.mean()), float(y_teste.mean())],
}, index=["treino", "teste"])
print(resumo_split.round(4).to_string())
print()
assert set(X_treino.index).isdisjoint(set(X_teste.index)), "Vazamento entre treino e teste."
assert len(X_treino) + len(X_teste) == len(X)
print("[OK] Conjuntos disjuntos e base totalmente alocada.")
print("Diferenca de proporcao treino/teste:",
      round(abs(float(y_treino.mean()) - float(y_teste.mean())), 6),
      "(estratificacao preservou o balanco)")

## 8. Treinamento de classificadores supervisionados

O treino é executado por `train.treinar_e_avaliar`, que usa **as mesmas** configurações do módulo
de linha de comando (`python -m src.cardioia.ml.train`):

| Modelo | Hiperparâmetros |
|---|---|
| **Regressão Logística** | `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))` |
| **Random Forest** | `RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)` |

**Por que esses dois?** A Regressão Logística é interpretável (coeficientes com direção e
magnitude) e serve de referência para dados tabulares; o Random Forest é não linear por construção
e representa bem regras com **limiares**, como as do rótulo.

**Mesmas condições:** mesmo `X_treino`/`y_treino`, mesma avaliação no mesmo `X_teste`.

> **Critério de escolha do modelo que será salvo — declarado ANTES de medir:**
> 1. maior **recall da classe `pico_risco=1`** (um falso negativo é o erro mais custoso);
> 2. desempate por **acurácia**;
> 3. persistindo o empate, o modelo mais **interpretável**.
>
> Nenhum ajuste de hiperparâmetros é feito depois de olhar o teste.

In [ ]:
# 8.1 - Treino pelo modulo (mesmo treino para os dois modelos)
execucao = train.treinar_e_avaliar(
    df=df,
    semente=SEMENTE,
    tamanho_teste=TAMANHO_TESTE,
    n_estimators=train.N_ESTIMATORS_PADRAO,
    raiz=RAIZ,
    origem_da_base=origem_dados,
)

print("Modelos treinados:")
for nome, modelo in execucao["modelos"].items():
    print(f"  [treinado] {nome:22s} | {type(modelo).__name__}")
print()
print("Hiperparametros efetivamente usados (lidos do objeto treinado):")
import json as _json
for nome, modelo in execucao["modelos"].items():
    print(f"  {nome}: {_json.dumps(train.descrever_parametros(modelo), ensure_ascii=False)}")
print()
print("Criterio de escolha:", execucao["metadados"]["criterio_de_escolha"])

## 9. Avaliação: acurácia, matriz de confusão e relatório de classificação

As métricas vêm de `evaluate.avaliar_modelo` (que usa apenas funções do `scikit-learn`) sobre o
conjunto de teste. A matriz de confusão segue o layout padrão:

```
                 previsto 0     previsto 1
real 0 (sem pico)   TN             FP
real 1 (pico)       FN             TP
```

Interpretação de domínio: **FN** = pico de risco não sinalizado (erro mais grave) e
**FP** = alarme sem pico (custo de recurso, não de segurança).

`evaluate` também trata explicitamente métricas indefinidas: se uma classe não aparecer em `y_true`
ou não for prevista, o valor vira `None` com um **aviso**, em vez de um `0.0` que esconderia o
problema.

In [ ]:
# 9.1 - Metricas no conjunto de teste (modulo evaluate)
for nome, avaliacao in execucao["avaliacoes"].items():
    print(evaluate.formatar_relatorio(avaliacao))
    print()

print("=" * 72)
print("Comparativo direto no MESMO conjunto de teste:")
comparativo = pandas.DataFrame([
    {
        "acuracia": avaliacao["acuracia"],
        "recall_classe_1": avaliacao["recall_classe_1"],
        "precisao_classe_1": avaliacao["precisao_classe_1"],
        "f1_classe_1": avaliacao["f1_classe_1"],
    }
    for avaliacao in execucao["avaliacoes"].values()
], index=list(execucao["avaliacoes"]))
print(comparativo.round(4).to_string())

### 9.2 Referência determinística (oráculo do material oficial)

Para saber **quanto do desempenho vem de aprendizado** e quanto vem da própria construção da base,
aplicamos a regra literal do material oficial (`risk_score >= 2`) ao conjunto de teste, usando
`dataset.rotulo_pela_regra`.

Isso **não é um modelo concorrente**: é a regra que *definiu* o rótulo, então o resultado dela é
tautologicamente perfeito. O valor aqui é servir de teto de referência e confirmar que a
implementação determinística em Python está correta — a mesma função que a etapa dos agentes usará
como verificação independente do modelo de ML.

In [ ]:
# 9.2 - Oracolo deterministico aplicado ao conjunto de teste
regra_no_teste = execucao["regra_deterministica_no_teste"]
avaliacao_da_regra = evaluate.avaliar_predicoes(
    y_teste, regra_no_teste, nome_do_modelo="Regra deterministica"
)
print(evaluate.formatar_relatorio(avaliacao_da_regra))
print()
print("Acuracia da regra (teto de referencia):", execucao["acuracia_da_regra"])
print("Observacao: acuracia = 1 e consequencia de o rotulo ter sido DEFINIDO por essa regra.")
print("O numero relevante e a distancia entre cada modelo e esse teto.")

## 10. Análise dos resultados e limitações

A célula 10.1 mostra **o que o modelo escolhido aprendeu** (importâncias/coeficientes) e os casos em
que ele erra. A 10.2 confronta o rótulo com os protocolos textuais da aula e imprime o resumo do
experimento.

**Pontos de leitura importantes**

1. **O rótulo é uma regra determinística** de 6 indicadores com limiares fixos (`risk_score >= 2`).
   Um modelo não linear (Random Forest) tende a reconstruir essa regra quase perfeitamente; a
   Regressão Logística, por ser linear, encontra um compromisso e erra mais na classe minoritária.
2. Portanto, **acurácia alta aqui não é evidência de capacidade preditiva real**: é evidência de que
   a regra foi reconstruída. O experimento é didático — aprendizado supervisionado sobre base
   sintética com rótulo conhecido — e não uma validação clínica.
3. **Desbalanceamento**: 26,58% de positivos. Reportamos recall/precisão/F1 da classe 1 porque a
   acurácia isolada esconde erros na classe de interesse.
4. **Sem vazamento**: `risk_score` não é feature (seção 6); treino e teste são disjuntos (seção 7).
5. **Limitação de interpretação causal**: as importâncias indicam associação dentro da base
   sintética, não causalidade clínica.
6. **Cobertura de casos extremos**: a base é gerada por distribuições com recorte (`clip`); há
   pouquíssimos casos com `risk_score` alto (5 ou 6 indicadores), o que limita a análise desses
   extremos.

In [ ]:
# 10.1 - Criterio pre-declarado aplicado + o que o modelo aprendeu
comparativo_ordenado = pandas.DataFrame(execucao["comparativo"]).set_index("modelo")
print("Ordenacao pelo criterio pre-declarado (1o recall da classe 1, 2o acuracia):")
print(comparativo_ordenado[[
    "acuracia", "recall_classe_1", "precisao_classe_1", "f1_classe_1", "interpretabilidade"
]].round(4).to_string())
modelo_escolhido_nome = execucao["nome_modelo_escolhido"]
modelo_escolhido = execucao["modelo_escolhido"]
print()
print("Modelo escolhido:", modelo_escolhido_nome)
print()

floresta = execucao["modelos"][train.NOME_MODELO_NAO_LINEAR]
importancias = pandas.Series(
    floresta.feature_importances_, index=COLUNAS_PREDITORAS
).sort_values(ascending=False)
print("Importancia das variaveis (Random Forest, impureza de Gini):")
print(importancias.round(4).to_string())
print()

regressao = execucao["modelos"][train.NOME_MODELO_INTERPRETAVEL].named_steps["logisticregression"]
coeficientes = pandas.Series(regressao.coef_[0], index=COLUNAS_PREDITORAS)
coeficientes = coeficientes.reindex(coeficientes.abs().sort_values(ascending=False).index)
print("Coeficientes da Regressao Logistica (features padronizadas; sinal = direcao do risco):")
print(coeficientes.round(4).to_string())
print()

nulas = importancias[importancias < 0.01]
if len(nulas):
    print("Variaveis praticamente ignoradas pelo Random Forest (< 1% de importancia):")
    print(nulas.round(6).to_string())
else:
    print("Nenhuma variavel ficou abaixo de 1% de importancia no Random Forest.")
print()

erros = X_teste.copy()
erros["pico_risco_real"] = y_teste.values
erros["pico_risco_previsto"] = modelo_escolhido.predict(X_teste)
erros["risk_score_da_regra"] = risk_score_recalculado.loc[X_teste.index].values
falsos_negativos = erros[(erros["pico_risco_real"] == 1) & (erros["pico_risco_previsto"] == 0)]
falsos_positivos = erros[(erros["pico_risco_real"] == 0) & (erros["pico_risco_previsto"] == 1)]

print(f"Erros do modelo escolhido ({modelo_escolhido_nome}):")
print("  falsos negativos (risco nao sinalizado):", len(falsos_negativos))
print("  falsos positivos (alarme sem risco real):", len(falsos_positivos))
if len(falsos_negativos):
    print()
    print("Exemplos de falsos negativos (com o risk_score da regra oficial):")
    print(falsos_negativos.head(5).round(2).to_string())

In [ ]:
# 10.2 - Divergencias entre o rotulo e os protocolos da aula + resumo do experimento
divergencias = pandas.DataFrame([
    ("spo2_media", "rotulo: < 92", "protocolo: < 88"),
    ("carga_sistema", "rotulo: > 80", "protocolo: > 85"),
    ("fc_media", "rotulo: > 110", "protocolo: > 120"),
    ("eventos_24h", "rotulo: > 4", "protocolo: > 5"),
    ("disponibilidade_recursos", "rotulo: < 40", "protocolo: < 30"),
    ("tempo_espera", "rotulo: > 60", "protocolo: > 90"),
    ("temperatura", "NAO entra no rotulo", "protocolo: > 38 (no gerador a variavel e N(24,5))"),
    ("densidade_eventos", "NAO entra no rotulo", "protocolo: > 6"),
    ("pa_sistolica", "NAO entra no rotulo", "protocolo: < 90"),
    ("risco sistemico", "rotulo: risk_score >= 2", "protocolo: 'risco sistemico alto' (sem limiar numerico)"),
], columns=["variavel", "limiar_no_rotulo_pico_risco", "limiar_no_protocolo_da_aula"])

print("Divergencias entre os limiares do rotulo e os protocolos textuais da aula:")
print(divergencias.to_string(index=False))
print()
print("Implicacao: a camada de ML (rotulo) e a camada de protocolos sao SEPARADAS.")
print("O modelo aprende o rotulo; os protocolos continuam sendo avaliados de forma")
print("deterministica em Python (secao 13). Um LLM nao decide esses limiares.")
print()
print(train.formatar_resumo(execucao))

## 11. Salvamento do modelo treinado

`train.salvar_artefatos` grava dois arquivos na pasta `models/` (mesma função usada pela CLI):

| Arquivo | Conteúdo |
|---|---|
| `modelo_pico_risco.pkl` | O modelo escolhido, serializado com `joblib`. |
| `modelo_pico_risco_metadata.json` | Metadados de rastreabilidade: features (e a **ordem**), semente, split, critério de escolha, parâmetros do estimador, limiar de decisão, métricas de teste, matriz de confusão, faixas observadas, versões das bibliotecas, SHA-256 da base usada e o aviso de dados sintéticos. |

O JSON é o que permite reutilizar o modelo com segurança (inclusive pelos agentes, na próxima
etapa): quem carregar o `.pkl` precisa saber exatamente quais colunas, em que ordem, o modelo
espera — e de onde vieram os números, sem inventar probabilidades.

In [ ]:
# 11.1 - Salvamento do modelo e dos metadados (modulo train)
artefatos = train.salvar_artefatos(execucao, pasta_modelos=PASTA_MODELOS, raiz=RAIZ)
caminho_modelo = artefatos["caminho_modelo"]
caminho_metadados = artefatos["caminho_metadados"]

print("Artefatos gravados:")
print("  modelo   :", caminho_modelo, "|", artefatos["tamanho_modelo_bytes"], "bytes")
print("  metadados:", caminho_metadados, "|", artefatos["tamanho_metadados_bytes"], "bytes")
print()
import json as _json
print("Metadados (sem o bloco de metricas, ja impresso na secao 9):")
print(_json.dumps(
    {chave: valor for chave, valor in artefatos["metadados"].items() if chave != "metricas_no_teste"},
    ensure_ascii=False, indent=2,
))

## 12. Carregamento do modelo salvo e teste de uma nova previsão

O artefato é reutilizado pela função `predict.prever_pico_risco`, que é a **mesma** interface que o
Agente Analista de Risco usará como *tool* na próxima etapa. Ela:

- carrega `models/modelo_pico_risco.pkl` e o JSON de metadados (erro claro se não existirem);
- valida o contrato de entrada (campos ausentes/desconhecidos, texto, booleano, `NaN`/`inf`,
  negativos) e **reordena** as colunas conforme o modelo espera;
- devolve `pico_risco_previsto`, `probabilidade_risco` (valor **cru**, sem arredondar),
  `classificacao_risco`, `modelo_utilizado` e `limiar_decisao`.

O limiar (0,5) é um parâmetro **operacional** do protótipo, documentado no metadata — **não** é um
limite clínico validado.

In [ ]:
# 12.1 - Reuso do artefato com a interface de previsao
modelo_recarregado = predict.carregar_modelo(pasta_modelos=PASTA_MODELOS)
metadados = predict.carregar_metadados(pasta_modelos=PASTA_MODELOS)
print("Modelo recarregado:", type(modelo_recarregado).__name__, "| nome no metadata:", metadados["nome_do_modelo"])

predicoes_em_memoria = modelo_escolhido.predict(X_teste)
predicoes_do_disco = modelo_recarregado.predict(X_teste)
identicas = bool((predicoes_em_memoria == predicoes_do_disco).all())
print("Previsoes identicas antes/depois do salvamento:", identicas)
assert identicas, "O modelo recarregado diverge do modelo em memoria."


def regra_do_rotulo(caso):
    # Implementacao deterministica (mesma de data/base_dados.py) via modulo dataset.
    ativos = dataset.indicadores_ativos(caso)
    risk_score = len(ativos)
    return risk_score, int(risk_score >= CORTE_RISK_SCORE), ativos


def apresentar_caso(caso, titulo):
    resultado = predict.prever_pico_risco(caso, pasta_modelos=PASTA_MODELOS)
    risk_score, pico_da_regra, ativos = regra_do_rotulo(caso)
    print(titulo)
    print("  entrada                :", caso)
    print("  probabilidade (crua)   :", resultado["probabilidade_risco"])
    print("  probabilidade (exibida):", round(resultado["probabilidade_risco"], 4))
    print("  pico_risco (modelo)    :", resultado["pico_risco_previsto"],
          "->", resultado["classificacao_risco"])
    print("  risk_score (regra)     :", risk_score, "-> pico_risco (regra):", pico_da_regra)
    print("  indicadores ativos     :", ativos if ativos else "nenhum")
    print("  modelo e regra concordam:", resultado["pico_risco_previsto"] == pico_da_regra)
    print("  avisos                 :", resultado["avisos"] if resultado["avisos"] else "nenhum")
    print()
    return resultado


caso_baixo = {
    "idade_media": 40, "carga_sistema": 45, "tempo_espera": 20, "fc_media": 78,
    "spo2_media": 98, "pa_sistolica": 118, "temperatura": 22,
    "densidade_eventos": 2, "eventos_24h": 1, "disponibilidade_recursos": 80,
}
caso_alto = {
    "idade_media": 70, "carga_sistema": 88, "tempo_espera": 65, "fc_media": 115,
    "spo2_media": 90, "pa_sistolica": 105, "temperatura": 30,
    "densidade_eventos": 5, "eventos_24h": 6, "disponibilidade_recursos": 35,
}

print("Teste de dois casos novos com o modelo RECARREGADO do disco")
print("=" * 72)
apresentar_caso(caso_baixo, "Caso A - cenario sem sobrecarga (esperado: sem pico de risco)")
apresentar_caso(caso_alto, "Cenario B - cenario critico (esperado: pico de risco)")

## 13. Simulação com os dados do paciente fornecidos pela aula

O caso vem de `data/novo_paciente.py`, lido por `dataset.carregar_caso_do_paciente_da_aula` (seção
4.3) — o arquivo original **não** é editado:

| Campo (como está no arquivo) | Valor |
|---|---:|
| `idade_media` | 62 |
| `carga_sistema` | 92 |
| `tempo_espera` | 75 |
| `fc_media` | 118 |
| `spo2_media` | 89 |
| `pa_sistolica` | 100 |
| `temperatura` | 37.5 |
| `densidade_eventos` | 7 |
| `eventos_24h` | 5 |
| `disponibilidade recursos` | 25 |

**Duas divergências do material, tratadas explicitamente:** o fragmento **não importa `pandas`** e a
última coluna está grafada com espaço (`disponibilidade recursos` → `disponibilidade_recursos`). A
normalização é feita pelo módulo e registrada na mensagem de erro quando alguém envia a grafia
original.

Além da previsão do modelo, esta seção roda a **camada determinística de protocolos** da base de
conhecimento da aula (10 regras) sobre o caso — em Python, não em LLM — e mostra o oráculo
`risk_score` para comparação.

In [ ]:
# 13.1 - Camada deterministica de protocolos (Python, sem LLM)
# Cada protocolo textual da aula e vinculado a UMA condicao numerica implementada aqui.
# O protocolo "risco sistemico alto" nao tem limiar numerico no material, por isso usa a
# saida do modelo como gatilho - isso fica explicito no resultado.
CONDICOES_PROTOCOLOS = [
    ("spo2 < 88", lambda caso: caso["spo2_media"] < 88),
    ("carga do sistema > 85", lambda caso: caso["carga_sistema"] > 85),
    ("fc > 120", lambda caso: caso["fc_media"] > 120),
    ("eventos_24h > 5", lambda caso: caso["eventos_24h"] > 5),
    ("disponibilidade_recursos < 30", lambda caso: caso["disponibilidade_recursos"] < 30),
    ("tempo_espera > 90", lambda caso: caso["tempo_espera"] > 90),
    ("temperatura > 38", lambda caso: caso["temperatura"] > 38),
    ("densidade_eventos > 6", lambda caso: caso["densidade_eventos"] > 6),
    ("pa sistólica < 90", lambda caso: caso["pa_sistolica"] < 90),
    ("risco sistêmico alto", None),
]

print("Vinculacao entre os protocolos oficiais e as condicoes implementadas:")
vinculacao = []
for protocolo in PROTOCOLOS_AULA:
    casados = [rotulo for rotulo, _ in CONDICOES_PROTOCOLOS if rotulo in protocolo.lower()]
    vinculacao.append((protocolo, len(casados)))
    print(f"  {len(casados)} condicao(oes) -> {protocolo}")
assert all(total == 1 for _, total in vinculacao), "Algum protocolo oficial nao tem (ou tem mais de uma) condicao implementada."
print("[OK] Todos os protocolos oficiais estao implementados deterministicamente.")


def protocolos_acionados(caso, pico_risco_previsto):
    # Aplica, em Python puro, todas as regras da base de conhecimento da aula.
    ativos = []
    for protocolo in PROTOCOLOS_AULA:
        for rotulo, condicao in CONDICOES_PROTOCOLOS:
            if rotulo not in protocolo.lower():
                continue
            if condicao is None:
                dispara = (pico_risco_previsto == 1)
            else:
                dispara = bool(condicao(caso))
            if dispara:
                ativos.append(protocolo)
            break
    return ativos

In [ ]:
# 13.2 - Previsao do paciente da aula: modelo salvo + oraculo + protocolos
resultado_paciente = predict.prever_pico_risco(caso_paciente, pasta_modelos=PASTA_MODELOS)
risk_score_paciente, pico_risco_da_regra, indicadores = regra_do_rotulo(caso_paciente)

print("=" * 72)
print("SIMULACAO - Paciente do material da aula")
print("=" * 72)
print("Entrada (valores de data/novo_paciente.py, coluna normalizada):")
for nome, valor in caso_paciente.items():
    print(f"  {nome:28s}: {valor}")
print()
print("Saida do modelo salvo (models/modelo_pico_risco.pkl):")
print("  modelo                  :", resultado_paciente["modelo_utilizado"],
      "|", resultado_paciente["classe_do_estimador"])
print("  probabilidade (crua)    :", resultado_paciente["probabilidade_risco"])
print("  probabilidade (exibida) :", round(resultado_paciente["probabilidade_risco"], 4))
print("  pico_risco previsto     :", resultado_paciente["pico_risco_previsto"],
      "->", resultado_paciente["classificacao_risco"])
print("  limiar e regra de decisao:", resultado_paciente["limiar_decisao"], "|",
      resultado_paciente["regra_de_decisao"])
print()
print("Verificacao deterministica (regra de data/base_dados.py):")
print("  indicadores ativos      :", indicadores)
print("  risk_score              :", risk_score_paciente, "de 6")
print("  pico_risco (regra)      :", pico_risco_da_regra)
print("  o modelo e a regra concordam:", resultado_paciente["pico_risco_previsto"] == pico_risco_da_regra)
print()
protocolos = protocolos_acionados(caso_paciente, resultado_paciente["pico_risco_previsto"])
print("Protocolos da base de conhecimento acionados (", len(protocolos), "):")
for protocolo in protocolos:
    print("  -", protocolo)
print()
print("Resumo pronto para um agente (formato de retorno de tool):")
print(" ", predict.resumo_para_agente(resultado_paciente))
print()
print("Observacao: o protocolo de temperatura (> 38) nao dispara porque a temperatura informada")
print("e 37.5 - e no gerador a variavel e N(24,5), ou seja, essa regra quase nunca e ativada")
print("pela base sintetica. Registro mantido por fidelidade ao material.")
print()
print("AVISO:", resultado_paciente["aviso_de_uso"])

## 14. Persistência dos artefatos no Google Colab

No Colab o sistema de arquivos é temporário: ao encerrar a sessão, `models/` é descartado. A célula
abaixo empacota os artefatos em `.zip` e dispara o download. Localmente, apenas lista o que foi
gravado no disco.

**Próximos passos do projeto (Etapa 3 — ainda não implementada nesta entrega)**

1. `src/knowledge/protocols.py` receberá a camada determinística de protocolos que aqui vive na
   seção 13.1 (mesmas regras, agora testáveis e reutilizáveis pelo Agente Especialista em Protocolos).
2. `src/schemas/risk_result.py` validará com Pydantic a saída de `predict.prever_pico_risco`
   (que já devolve JSON serializável) antes de virar resposta ao usuário.
3. `predict.prever_pico_risco` e `predict.resumo_para_agente` serão expostos como *tools* do OpenAI
   Agents SDK; `dataset.rotulo_pela_regra` continuará sendo a verificação determinística, **nunca**
   substituída por decisão de LLM.

In [ ]:
# 14.1 - Empacotamento/download dos artefatos + resumo final
from zipfile import ZipFile

arquivos_gerados = sorted(caminho for caminho in PASTA_MODELOS.glob("*") if caminho.is_file())
print("Artefatos em", PASTA_MODELOS, ":")
for caminho in arquivos_gerados:
    print(f"  {caminho.name:38s} {caminho.stat().st_size:>8d} bytes")
print()

if EM_COLAB:
    caminho_zip = (PASTA_MODELOS.parent / "cardioia_fase6_modelo.zip").resolve()
    with ZipFile(caminho_zip, "w") as pacote:
        for caminho in arquivos_gerados:
            pacote.write(caminho, arcname=f"models/{caminho.name}")
    print("Pacote criado:", caminho_zip, "|", caminho_zip.stat().st_size, "bytes")
    print("Iniciando o download...")
    from google.colab import files  # apenas disponivel no Colab
    files.download(str(caminho_zip))
else:
    print("Execucao local: os artefatos permanecem em", PASTA_MODELOS)
    print("Para treinar pela linha de comando: python -m src.cardioia.ml.train")
    print("Para rodar os testes:               python -m pytest tests -q")

print()
print("=" * 72)
print("Resumo final desta execucao")
print("=" * 72)
print("  base                     :", origem_dados)
print("  sha256 da base           :", dataset.sha256_da_base(df))
print("  features x amostras      :", len(COLUNAS_PREDITORAS), "x", len(df))
print("  modelo salvo             :", execucao["nome_modelo_escolhido"])
for nome, avaliacao in execucao["avaliacoes"].items():
    print(f"  {nome:24s} : acuracia={avaliacao['acuracia']} | recall(1)={round(avaliacao['recall_classe_1'], 4)}")
print(f"  {'Regra deterministica':24s} : acuracia={execucao['acuracia_da_regra']} (teto, por construcao)")
print("  previsao do paciente     :", resultado_paciente["pico_risco_previsto"],
      "(probabilidade", round(resultado_paciente["probabilidade_risco"], 4), ")")
print("  protocolos acionados     :", len(protocolos))
print()
print("AVISO: base sintetica; este modelo nao tem validacao clinica.")